# Scoring Automático v2.0 — Mes 5

## 0. Setup — datos del Analizador

In [18]:
# --- SETUP: DATOS DEL ANALIZADOR ---

import pandas as pd
import numpy as np

# ── Datos del Mes 3 — dataset temporal ─────────────────────
datos_temporales = {
    'nombre': ['Ferretería García']*3 + ['Bar El Rincón']*3 +
              ['Peluquería Ana']*3 + ['Tienda Moda Sol']*3 +
              ['Gestoría Pérez']*3 + ['Clínica Dental Martínez']*3 +
              ['Academia Idiomas Sol']*3 + ['Clínica Dental García']*3 +
              ['Academia Idiomas English House']*3,
    'año': [2022, 2023, 2024] * 9,
    'facturacion': [
        295000, 308000, 320000, 165000, 172000, 180000,
        85000,  90000,  95000,  195000, 202000, 210000,
        125000, 133000, 140000, 255000, 268000, 280000,
        145000, 152000, 160000, 270000, 283000, 295000,
        138000, 146000, 155000,
    ],
    'margen_bruto_pct': [
        32.5, 33.1, 34.4, 46.8, 47.0, 47.2,
        59.2, 59.5, 60.0, 32.5, 32.9, 33.3,
        66.5, 67.2, 67.9, 59.0, 59.5, 60.0,
        69.5, 69.7, 70.0, 59.5, 59.8, 60.0,
        49.5, 49.7, 50.0,
    ]
}
df_temporal = pd.DataFrame(datos_temporales)
df_temporal['fecha'] = pd.to_datetime(df_temporal['año'].astype(str) + '-01-01')
df_temporal = df_temporal.set_index('fecha')
crecimiento = df_temporal.groupby('nombre')['facturacion'].pct_change() * 100
df_temporal['crecimiento_pct'] = crecimiento.round(1)

# ── Datos del Mes 2 — ratios y scoring ─────────────────────
datos = {
    'nombre': ['Ferretería García', 'Bar El Rincón', 'Peluquería Ana',
               'Tienda Moda Sol', 'Gestoría Pérez', 'Clínica Dental Martínez',
               'Academia Idiomas Sol', 'Clínica Dental García',
               'Academia de Idiomas English House'],
    'sector': ['Ferretería', 'Hostelería', 'Peluquería', 'Moda', 'Gestoría',
               'Salud', 'Educación', 'Salud', 'Educación'],
    'facturacion': [320000, 180000, 95000, 210000, 140000,
                    280000, 160000, 295000, 155000],
    'coste_ventas': [210000, 95000, 38000, 140000, 45000,
                     84000, 48000, 118000, 77500],
    'empleados': [5, 4, 2, 3, 3, 6, 5, 6, 5],
    'deuda': [45000, 12000, 8000, 35000, 5000,
              90000, 15000, 95000, 14000],
    'activo_total': [180000, 85000, 40000, 120000, 70000,
                     220000, 75000, 230000, 72000],
    'año_fundacion': [2008, 2015, 2019, 2012, 2006,
                      2014, 2016, 2013, 2017]
}
df = pd.DataFrame(datos)
df['margen_bruto_pct'] = ((df['facturacion'] - df['coste_ventas']) /
                           df['facturacion'] * 100).round(1)
df['ratio_endeudamiento'] = (df['deuda'] / df['activo_total']).round(3)
df['fact_por_empleado'] = (df['facturacion'] / df['empleados']).round(0)
df['años_activo'] = 2024 - df['año_fundacion']

# Score estadístico por percentiles (60% margen, 40% deuda inversa)
df['pct_margen'] = df['margen_bruto_pct'].rank(pct=True).round(2)
df['pct_deuda'] = (1 - df['ratio_endeudamiento']).rank(pct=True).round(2)
df['score_estadistico'] = (df['pct_margen'] * 0.60 +
                            df['pct_deuda'] * 0.40).round(3)

print('✅ Setup completado')
print(f'   df_temporal: {len(df_temporal)} filas')
print(f'   df: {len(df)} empresas con ratios y scoring')

✅ Setup completado
   df_temporal: 27 filas
   df: 9 empresas con ratios y scoring


## 1. ¿Qué es un problema de clasificación?

In [19]:
# --- BLOQUE 1: PREPARACIÓN DEL DATASET PARA ML ---

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# --- PREPARAR EL DATASET ---
# Usamos los datos del Mes 3: df_temporal con datos de 2022-2024
# Nos quedamos con los datos de 2024 — el snapshot más reciente
df_ml = df_temporal[df_temporal['año'] == 2024].copy()
df_ml = df_ml.set_index('nombre')

# Añadir variables del Mes 2 (scores, ratios)
# Asegúrate de que df con los ratios del Mes 2 está disponible
for col in ['margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado',
            'años_activo', 'score_estadistico', 'riesgo_deuda']:
    if col in df.columns:
        df_ml[col] = df.set_index('nombre')[col]

# --- VARIABLE OBJETIVO (TARGET) ---
# La etiqueta que el modelo aprenderá a predecir
# TOP = empresa en el cuartil superior del score → clase 1
# MID/LOW = empresa en el resto → clase 0
umbral_top = df_ml['score_estadistico'].quantile(0.67)
df_ml['es_top'] = (df_ml['score_estadistico'] >= umbral_top).astype(int)

print('=== DATASET PARA ML ===')
print(df_ml[['margen_bruto_pct', 'ratio_endeudamiento', 'crecimiento_pct',
             'score_estadistico', 'es_top']])
print(f'\nEmpresas TOP: {df_ml["es_top"].sum()}')
print(f'Empresas MID/LOW: {(df_ml["es_top"]==0).sum()}')

=== DATASET PARA ML ===
                                margen_bruto_pct  ratio_endeudamiento  \
nombre                                                                  
Ferretería García                           34.4                0.250   
Bar El Rincón                               47.2                0.141   
Peluquería Ana                              60.0                0.200   
Tienda Moda Sol                             33.3                0.292   
Gestoría Pérez                              67.9                0.071   
Clínica Dental Martínez                     70.0                0.409   
Academia Idiomas Sol                        70.0                0.200   
Clínica Dental García                       60.0                0.413   
Academia Idiomas English House               NaN                  NaN   

                                crecimiento_pct  score_estadistico  es_top  
nombre                                                                      
Ferretería García 

## 2. Features y split train-test

In [20]:
# --- BLOQUE 2: FEATURES Y SPLIT TRAIN-TEST ---

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- SELECCIÓN DE FEATURES ---
# Las variables que el modelo usará para aprender
# No incluyes score_estadistico ni es_top — son la variable objetivo
features = [
    'margen_bruto_pct',       # margen bruto
    'ratio_endeudamiento',    # nivel de deuda
    'fact_por_empleado',      # eficiencia
    'años_activo',            # antigüedad
    'crecimiento_pct',        # crecimiento reciente
    'facturacion',            # tamaño del negocio
]

# Filtrar solo las filas sin NaN en las features
df_limpio = df_ml[features + ['es_top']].dropna()

X = df_limpio[features]   # variables de entrada
y = df_limpio['es_top']   # variable objetivo

print(f'Empresas disponibles: {len(X)}')
print(f'Features: {features}')
print(f'\nDistribución de clases:')
print(y.value_counts())

# --- SPLIT TRAIN-TEST ---
# Con solo 9 empresas usamos test_size pequeño
# En producción lo normal es 0.2-0.3
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=y if y.sum() >= 2 else None
)

print(f'\nTrain: {len(X_train)} empresas')
print(f'Test:  {len(X_test)} empresas')

# --- ESCALADO ---
# StandardScaler normaliza cada feature a media 0 y std 1
# Necesario para algunos modelos (no para XGBoost, pero es buena práctica)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nFeatures escaladas correctamente')

Empresas disponibles: 8
Features: ['margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado', 'años_activo', 'crecimiento_pct', 'facturacion']

Distribución de clases:
es_top
0    5
1    3
Name: count, dtype: int64

Train: 6 empresas
Test:  2 empresas

Features escaladas correctamente


## 3. Primer modelo — Regresión Logística

In [21]:
# --- BLOQUE 3: REGRESIÓN LOGÍSTICA ---

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- ENTRENAR EL MODELO ---
modelo_lr = LogisticRegression(random_state=42, max_iter=1000)
modelo_lr.fit(X_train_sc, y_train)

# --- PREDICCIONES ---
y_pred_lr = modelo_lr.predict(X_test_sc)
y_prob_lr = modelo_lr.predict_proba(X_test_sc)[:, 1]  # probabilidad de ser TOP

# --- MÉTRICAS DE EVALUACIÓN ---
print('=== REGRESIÓN LOGÍSTICA — RESULTADOS ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.2f}')
print('\nInforme de clasificación:')
print(classification_report(y_test, y_pred_lr,
                             target_names=['MID/LOW', 'TOP']))

# --- MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred_lr)
print('Matriz de confusión:')
print(f'Verdaderos positivos (TOP correctamente clasificadas): {cm[1,1]}')
print(f'Falsos positivos (MID/LOW clasificadas como TOP): {cm[0,1]}')
print(f'Falsos negativos (TOP clasificadas como MID/LOW): {cm[1,0]}')

# --- COEFICIENTES ---
coef_df = pd.DataFrame({
    'feature': features,
    'coeficiente': modelo_lr.coef_[0].round(3)
}).sort_values('coeficiente', ascending=False)
print('\nImportancia de variables (coeficientes):')
print(coef_df.to_string())

=== REGRESIÓN LOGÍSTICA — RESULTADOS ===
Accuracy: 0.50

Informe de clasificación:
              precision    recall  f1-score   support

     MID/LOW       0.50      1.00      0.67         1
         TOP       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

Matriz de confusión:
Verdaderos positivos (TOP correctamente clasificadas): 0
Falsos positivos (MID/LOW clasificadas como TOP): 0
Falsos negativos (TOP clasificadas como MID/LOW): 1

Importancia de variables (coeficientes):
               feature  coeficiente
0     margen_bruto_pct        0.802
1  ratio_endeudamiento        0.222
5          facturacion        0.171
4      crecimiento_pct        0.143
3          años_activo        0.059
2    fact_por_empleado       -0.657


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 4. Random Forest — segundo modelo de referencia

In [22]:
# --- BLOQUE 4: RANDOM FOREST ---

from sklearn.ensemble import RandomForestClassifier

# --- RANDOM FOREST ---
modelo_rf = RandomForestClassifier(
    n_estimators=100,   # número de árboles
    max_depth=3,        # profundidad máxima — limita el overfitting
    random_state=42
)
modelo_rf.fit(X_train_sc, y_train)

y_pred_rf = modelo_rf.predict(X_test_sc)
y_prob_rf = modelo_rf.predict_proba(X_test_sc)[:, 1]

print('=== RANDOM FOREST — RESULTADOS ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.2f}')
print(classification_report(y_test, y_pred_rf,
                             target_names=['MID/LOW', 'TOP']))

# --- IMPORTANCIA DE VARIABLES ---
importancia_rf = pd.DataFrame({
    'feature': features,
    'importancia': modelo_rf.feature_importances_.round(3)
}).sort_values('importancia', ascending=False)

print('\nImportancia de variables (Random Forest):')
print(importancia_rf.to_string())

# --- COMPARATIVA LR vs RF ---
print('\n=== COMPARATIVA DE MODELOS ===')
comparativa = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr),
                 accuracy_score(y_test, y_pred_rf)],
}).round(3)
print(comparativa.to_string())
print('\n→ El mejor modelo de esta semana pasa a la Semana 2 para competir con XGBoost')

=== RANDOM FOREST — RESULTADOS ===
Accuracy: 1.00
              precision    recall  f1-score   support

     MID/LOW       1.00      1.00      1.00         1
         TOP       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Importancia de variables (Random Forest):
               feature  importancia
0     margen_bruto_pct        0.347
2    fact_por_empleado        0.232
3          años_activo        0.124
5          facturacion        0.112
4      crecimiento_pct        0.093
1  ratio_endeudamiento        0.092

=== COMPARATIVA DE MODELOS ===
                Modelo  Accuracy
0  Regresión Logística       0.5
1        Random Forest       1.0

→ El mejor modelo de esta semana pasa a la Semana 2 para competir con XGBoost


## 5. Cross-validation — evaluación honesta

In [23]:
# --- BLOQUE 5: CROSS-VALIDATION ---

from sklearn.model_selection import cross_val_score, StratifiedKFold

# --- CROSS-VALIDATION ---
# Con 8 empresas usamos 4 folds — cada fold tiene 2 empresas en test
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# --- CROSS-VAL PARA REGRESIÓN LOGÍSTICA ---
scores_lr = cross_val_score(
    modelo_lr, X_train_sc, y_train,
    cv=cv, scoring='f1_weighted'
)

# --- CROSS-VAL PARA RANDOM FOREST ---
scores_rf = cross_val_score(
    modelo_rf, X_train_sc, y_train,
    cv=cv, scoring='f1_weighted'
)

print('=== CROSS-VALIDATION — RESULTADOS ===')
print(f'Regresión Logística — F1 medio: {scores_lr.mean():.3f} (±{scores_lr.std():.3f})')
print(f'Random Forest       — F1 medio: {scores_rf.mean():.3f} (±{scores_rf.std():.3f})')

print(f'\nScores por fold — Regresión Logística: {scores_lr.round(2)}')
print(f'Scores por fold — Random Forest:       {scores_rf.round(2)}')

# --- CONCLUSIÓN ---
mejor = 'Random Forest' if scores_rf.mean() > scores_lr.mean() \
        else 'Regresión Logística'
print(f'\nMejor modelo esta semana: {mejor}')
print(f'→ Pasa a la Semana 2 para competir con XGBoost')

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=4.
  warnings.warn(


=== CROSS-VALIDATION — RESULTADOS ===
Regresión Logística — F1 medio: 0.333 (±0.408)
Random Forest       — F1 medio: 0.667 (±0.333)

Scores por fold — Regresión Logística: [0.   0.33 1.   0.  ]
Scores por fold — Random Forest:       [0.33 0.33 1.   1.  ]

Mejor modelo esta semana: Random Forest
→ Pasa a la Semana 2 para competir con XGBoost
